# Last.fm Album Recommendations

Similar-album recommendations via Last.fm's `track.getSimilar` API.

Compare three ways of picking the seed track from the album:

- **(a) Top listener** — the track with the most Last.fm listeners on the album
- **(b) Random** — a random track from the album tracklist
- **(c) All-tracks overlap** — run similarity for every album track and return albums recommended by multiple tracks; falls back to **(a)** if there is no overlap

All paths exclude albums by the seed artist. API responses are cached per run.

Set your API key in `bench/.env`:

```
LASTFM_API_KEY=your_key_here
```

In [64]:
import importlib
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

import lastfm_albums
importlib.reload(lastfm_albums)
from lastfm_albums import DEFAULT_FETCH_FLOOR, get_album_info, recommend_album, set_request_delay

In [65]:
DATA_DIR = Path("../datasets/")

## Seed album

In [66]:
SEED_ALBUM = "OK Computer"
SEED_ARTIST = "Radiohead"
N_RECS = 1          # recommendations to return per album
FETCH_FLOOR = 10    # similar tracks to fetch/resolve (filters often shrink the pool)
RANDOM_SEED = 42

seed = get_album_info(SEED_ARTIST, SEED_ALBUM)
pd.Series({k: seed[k] for k in seed if k != "tracks"})

key                                   radiohead::ok computer
artist                                             Radiohead
album                                            OK Computer
mbid                    0b6b4ba0-d36f-47bd-b4ea-6a5b91842d29
playcount                                          260565863
listeners                                            4829400
url          https://www.last.fm/music/Radiohead/OK+Computer
dtype: object

## Seed-track strategies

Run one cell at a time. Each calls a single strategy via `recommend_album()`.

In [67]:
def show_strategy(title, seed_track, recommendations, extra_cols=None):
    print(title)
    if seed_track is not None:
        print(f"Seed track: {seed_track['name']}", end="")
        if "listeners" in seed_track:
            print(f" ({seed_track['listeners']:,} listeners)")
        else:
            print()
    if recommendations is None or recommendations.empty:
        print("(no results)")
        return
    cols = ["album", "artist", "similarity_score", "matched_via", "url"]
    if extra_cols:
        cols = extra_cols + cols
    display(recommendations[cols])


def run_strategy(track_selection, *, clear_cache=True):
    """Run one seed-track strategy. Returns (seed, seed_track, recs, used_fallback)."""
    try:
        return recommend_album(
            SEED_ALBUM,
            artist=SEED_ARTIST,
            n_recs=N_RECS,
            fetch_floor=FETCH_FLOOR,
            track_selection=track_selection,
            random_seed=RANDOM_SEED,
            clear_cache=clear_cache,
        )
    except Exception as exc:
        print(f"Error: {exc}")
        return None, None, pd.DataFrame(), False

In [68]:
seed, seed_track_a, recs_a, _ = run_strategy("top_listener")
show_strategy("(a) Top-listener track", seed_track_a, recs_a)

(a) Top-listener track
Seed track: No Surprises (3,412,517 listeners)


,album,artist,similarity_score,matched_via,url
0,Surfer Rosa,Pixies,0.275238,track 'Where Is My Mind?' similar to 'No Surpr...,https://www.last.fm/music/Pixies/Surfer+Rosa


In [69]:
seed, seed_track_b, recs_b, _ = run_strategy("random", clear_cache=False)
show_strategy("(b) Random track", seed_track_b, recs_b)

(b) Random track
Seed track: Lucky


,album,artist,similarity_score,matched_via,url
0,Grace,Jeff Buckley,0.162629,track 'Grace' similar to 'Lucky',https://www.last.fm/music/Jeff+Buckley/Grace


In [ ]:
seed, seed_track_c, recs_c, used_fallback_c = run_strategy("all_tracks_overlap", clear_cache=False)

title = "(c) All-tracks overlap"
if used_fallback_c:
    title += " (no overlap — fell back to top-listener track)"
extra = ["vote_count"] if "vote_count" in recs_c.columns else None
show_strategy(title, seed_track_c, recs_c, extra_cols=extra)

In [ ]:
# Optional: overlap between (a) and (b). Run those cells first.
if "recs_a" in globals() and "recs_b" in globals() and not recs_a.empty and not recs_b.empty:
    display(
        recs_a.merge(recs_b, on="key", suffixes=("_top_listener", "_random"))[
            [
                "album_top_listener",
                "artist_top_listener",
                "seed_track_top_listener",
                "similarity_score_top_listener",
                "similarity_score_random",
            ]
        ]
    )
else:
    print("Run the (a) and (b) cells first.")

In [ ]:
SEED_ALBUM = "Acabou Chorare"
SEED_ARTIST = "Novos Baianos"

SEED_ALBUM = "Appetite for Destruction"
SEED_ARTIST = "Guns N' Roses"

seed, seed_track_b, recs_b, _ = run_strategy("random")
show_strategy("(b) Random track", seed_track_b, recs_b)